## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    average_precision_score, roc_auc_score,
    ConfusionMatrixDisplay, PrecisionRecallDisplay
)

import lightgbm as lgb
import optuna
import shap

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RANDOM_STATE = 42
N_FOLDS = 5
np.random.seed(RANDOM_STATE)

---
## 2. Load Data + LLM Features

In [ ]:
df = pd.read_csv(Path('../../data/claims_cleaned.csv'))
llm_df = pd.read_csv(Path('../../data/llm_features.csv'))

print(f'Claims: {df.shape}, LLM features: {llm_df.shape}')
print(f'\nLLM feature columns: {[c for c in llm_df.columns if c != "row_idx"]}')

In [ ]:
# Inspect LLM feature distributions
llm_feature_cols = [c for c in llm_df.columns if c not in ['row_idx', 'error']]

for col in llm_feature_cols:
    print(f'\n--- {col} ---')
    print(llm_df[col].value_counts())

In [ ]:
# Check LLM feature vs target relationship
llm_df['target'] = df['target'].values

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
for ax, col in zip(axes.flat, llm_feature_cols):
    ct = pd.crosstab(llm_df[col], llm_df['target'], normalize='index')
    if 0 in ct.columns:
        ct[0].sort_values(ascending=False).plot(kind='bar', ax=ax, color='salmon')
    ax.set_title(f'Decline Rate by {col}', fontsize=10)
    ax.set_ylabel('Decline Rate')
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('LLM-Extracted Features vs Decline Rate', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Encode LLM Features + Build Feature Matrix

In [ ]:
# Encode LLM features
llm_encoded = pd.DataFrame()
llm_encoders = {}

for col in llm_feature_cols:
    le = LabelEncoder()
    llm_encoded[f'llm_{col}'] = le.fit_transform(llm_df[col].fillna('unknown').astype(str))
    llm_encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

llm_encoded_cols = list(llm_encoded.columns)
print(f'\nEncoded LLM features: {len(llm_encoded_cols)}')

In [ ]:
# Structural features (same as v2/v3)
DROP_FEATURES = [
    'deviceCost', 'balance_change', 'has_balance_change', 'oldBalanceRRP',
    'balanceRRP', 'smashed', 'frontOrBackCamera', 'other',
    'relationship_encoded', 'buttons', 'connection', 'charging',
]
with open(Path('../../data/feature_columns.txt'), 'r') as f:
    all_features = [line.strip() for line in f.readlines()]
structural_features = [f for f in all_features if f not in DROP_FEATURES]

# Combine: structural + LLM features
feature_cols = structural_features + llm_encoded_cols

for col in llm_encoded_cols:
    df[col] = llm_encoded[col].values

X = df[feature_cols].fillna(0).values
y = df['target'].values

print(f'Structural features: {len(structural_features)}')
print(f'LLM features:        {len(llm_encoded_cols)}')
print(f'Total features:      {len(feature_cols)}')
print(f'X shape: {X.shape}')

---
## 4. Evaluation Helpers

In [ ]:
def evaluate_cv(model, X, y, model_name='Model', n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    all_true, all_pred, all_proba = [], [], []
    fold_metrics = []
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[va_idx])
        y_proba = model.predict_proba(X[va_idx])[:, 1]
        y_val = y[va_idx]
        
        fold_metrics.append({
            'fold': fold+1,
            'f1_macro': f1_score(y_val, y_pred, average='macro'),
            'f1_declined': f1_score(y_val, y_pred, pos_label=0),
            'pr_auc': average_precision_score(y_val, y_proba),
            'roc_auc': roc_auc_score(y_val, y_proba),
        })
        all_true.extend(y_val); all_pred.extend(y_pred); all_proba.extend(y_proba)
    
    mdf = pd.DataFrame(fold_metrics)
    all_true, all_pred, all_proba = np.array(all_true), np.array(all_pred), np.array(all_proba)
    
    print(f'\n{"="*60}')
    print(f'{model_name} — {n_folds}-Fold CV')
    print(f'{"="*60}')
    print(mdf.to_string(index=False))
    print(f'\nMean +/- Std:')
    for col in ['f1_macro', 'f1_declined', 'pr_auc', 'roc_auc']:
        print(f'  {col:15s}: {mdf[col].mean():.4f} +/- {mdf[col].std():.4f}')
    print(f'\n{classification_report(all_true, all_pred, target_names=["Declined", "Completed"])}')
    
    return {'metrics_df': mdf, 'y_true': all_true, 'y_pred': all_pred, 'y_proba': all_proba}


def plot_evaluation(results, model_name='Model'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ConfusionMatrixDisplay(confusion_matrix(results['y_true'], results['y_pred']),
                           display_labels=['Declined', 'Completed']).plot(ax=axes[0], cmap='Blues')
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    PrecisionRecallDisplay.from_predictions(results['y_true'], results['y_proba'], ax=axes[1], name=model_name)
    axes[1].set_title(f'{model_name} — PR Curve')
    for label, name in [(0, 'Declined'), (1, 'Completed')]:
        mask = results['y_true'] == label
        axes[2].hist(results['y_proba'][mask], bins=30, alpha=0.5, label=name, density=True)
    axes[2].set_xlabel('P(Completed)'); axes[2].legend()
    axes[2].set_title(f'{model_name} — Probability Distribution')
    axes[2].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()

---
## 5. Model A — LightGBM with Structural + LLM Features (Default)

In [ ]:
lgb_default = lgb.LGBMClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgb_default_results = evaluate_cv(lgb_default, X, y, 'LGB + Structural + LLM (Default)')
plot_evaluation(lgb_default_results, 'LGB + Structural + LLM (Default)')

---
## 6. Model B — LLM Features Only (How Much Signal Do They Carry?)

In [ ]:
X_llm_only = df[llm_encoded_cols].values

lgb_llm_only = lgb.LGBMClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgb_llm_only_results = evaluate_cv(lgb_llm_only, X_llm_only, y, 'LGB + LLM Features Only')
plot_evaluation(lgb_llm_only_results, 'LGB + LLM Features Only')

---
## 7. Model C — Optuna Tuned (Balanced Objective)

In [ ]:
def lgb_objective_v4(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[va_idx])
        f1_m = f1_score(y[va_idx], y_pred, average='macro')
        f1_d = f1_score(y[va_idx], y_pred, pos_label=0)
        scores.append(0.5 * f1_m + 0.5 * f1_d)
    return np.mean(scores)

study = optuna.create_study(direction='maximize', study_name='lgb_v4')
study.optimize(lgb_objective_v4, n_trials=80, show_progress_bar=True)

print(f'Best score: {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
lgb_tuned = lgb.LGBMClassifier(
    **study.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgb_tuned_results = evaluate_cv(lgb_tuned, X, y, 'LGB + Struct + LLM (Tuned v4)')
plot_evaluation(lgb_tuned_results, 'LGB + Struct + LLM (Tuned v4)')

---
## 8. Full Comparison — All Attempts

In [ ]:
prev = pd.read_csv(Path('../../models/model_comparison_v1_v2_v3.csv'), index_col=0)

v4_entries = [
    ('v4: LGB + Struct + LLM', lgb_default_results),
    ('v4: LGB + LLM Only', lgb_llm_only_results),
    ('v4: LGB + Struct + LLM (Tuned)', lgb_tuned_results),
]

v4_rows = []
for name, res in v4_entries:
    mdf = res['metrics_df']
    v4_rows.append({
        'Model': name,
        'F1 Macro (mean)': mdf['f1_macro'].mean(),
        'F1 Macro (std)': mdf['f1_macro'].std(),
        'F1 Declined (mean)': mdf['f1_declined'].mean(),
        'PR-AUC (mean)': mdf['pr_auc'].mean(),
        'ROC-AUC (mean)': mdf['roc_auc'].mean(),
    })
v4_df = pd.DataFrame(v4_rows).set_index('Model').round(4)

all_comparison = pd.concat([prev, v4_df])
print('=== FULL MODEL COMPARISON (v1 / v2 / v3 / v4) ===')
print(all_comparison.to_string())

In [ ]:
# Best model per metric
print(f'\nBest F1 Macro:    {all_comparison["F1 Macro (mean)"].idxmax()} = {all_comparison["F1 Macro (mean)"].max():.4f}')
print(f'Best F1 Declined: {all_comparison["F1 Declined (mean)"].idxmax()} = {all_comparison["F1 Declined (mean)"].max():.4f}')
print(f'Best PR-AUC:      {all_comparison["PR-AUC (mean)"].idxmax()} = {all_comparison["PR-AUC (mean)"].max():.4f}')

In [ ]:
# Visual: progression across attempts
highlight = all_comparison.loc[[
    'v1: LightGBM (Default)',
    'v2: LGB + Emb (Default)',
    'v3: BERT hybrid + LGB',
    'v4: LGB + Struct + LLM',
    'v4: LGB + Struct + LLM (Tuned)',
]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#4C78A8', '#54A24B', '#E45756', '#F58518', '#B07AA1']

for ax, metric in zip(axes, ['F1 Macro (mean)', 'F1 Declined (mean)', 'PR-AUC (mean)']):
    highlight[metric].plot(kind='bar', ax=ax, color=colors, edgecolor='black')
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=35)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Progression: v1 → v2 → v3 → v4', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Threshold Optimization

In [ ]:
# Pick best v4 model
v4_all = {'Default': lgb_default_results, 'LLM Only': lgb_llm_only_results, 'Tuned': lgb_tuned_results}
best_v4_key = max(v4_all, key=lambda k: v4_all[k]['metrics_df']['f1_macro'].mean())
best_v4 = v4_all[best_v4_key]
print(f'Threshold tuning on: {best_v4_key}')

y_true, y_proba = best_v4['y_true'], best_v4['y_proba']

thresholds = np.arange(0.1, 0.95, 0.01)
f1_macros = [f1_score(y_true, (y_proba >= t).astype(int), average='macro') for t in thresholds]
f1_declineds = [f1_score(y_true, (y_proba >= t).astype(int), pos_label=0) for t in thresholds]

best_t = thresholds[np.argmax(f1_macros)]

print(f'Best threshold: {best_t:.2f} -> F1 Macro={max(f1_macros):.4f}')
print(f'Default (0.50): F1 Macro={f1_score(y_true, (y_proba >= 0.5).astype(int), average="macro"):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_macros, label='F1 Macro', linewidth=2)
ax.plot(thresholds, f1_declineds, label='F1 Declined', linewidth=2)
ax.axvline(x=best_t, color='red', linestyle=':', label=f'Best ({best_t:.2f})')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Threshold'); ax.set_ylabel('Score'); ax.legend()
ax.set_title('Threshold Optimization'); plt.tight_layout(); plt.show()

print(f'\n=== Report at Threshold {best_t:.2f} ===')
print(classification_report(y_true, (y_proba >= best_t).astype(int), target_names=['Declined', 'Completed']))

---
## 10. SHAP Explainability

In [ ]:
# Train final model on full data
final_model = lgb.LGBMClassifier(
    **study.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
final_model.fit(X, y)

# SHAP
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values
print(f'SHAP shape: {shap_vals.shape}')

In [ ]:
# SHAP summary
plt.figure(figsize=(12, 12))
shap.summary_plot(shap_vals, X, feature_names=feature_cols, show=False, max_display=30)
plt.title('SHAP Summary — v4 (Structural + LLM Features)')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP bar
plt.figure(figsize=(10, 10))
shap.summary_plot(shap_vals, X, feature_names=feature_cols, plot_type='bar', show=False, max_display=30)
plt.title('SHAP Feature Importance — v4')
plt.tight_layout(); plt.show()

In [ ]:
# Contribution: structural vs LLM features
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_by_feature = pd.Series(mean_abs_shap, index=feature_cols)

llm_shap = shap_by_feature[llm_encoded_cols].sum()
struct_shap = shap_by_feature[structural_features].sum()
total = llm_shap + struct_shap

print(f'SHAP contribution:')
print(f'  Structural: {struct_shap:.4f} ({struct_shap/total:.1%})')
print(f'  LLM:        {llm_shap:.4f} ({llm_shap/total:.1%})')

# Top LLM features
print(f'\nTop LLM features by SHAP:')
print(shap_by_feature[llm_encoded_cols].sort_values(ascending=False))

---
## 11. Save Artifacts

In [ ]:
output_dir = Path('../../models')

# Model
joblib.dump(final_model, output_dir / 'lgb_claim_model_v4.joblib')

# SHAP
shap_df = pd.DataFrame(shap_vals, columns=feature_cols)
shap_df.to_csv(output_dir / 'shap_values_v4.csv', index=False)

# Config
config = {
    'optimal_threshold': float(best_t),
    'feature_columns': feature_cols,
    'structural_features': structural_features,
    'llm_features': llm_encoded_cols,
    'llm_model': 'claude-haiku-4-5-20251001',
    'best_params': study.best_params,
}
with open(output_dir / 'model_config_v4.json', 'w') as f:
    json.dump(config, f, indent=2)

# Full comparison
all_comparison.to_csv(output_dir / 'model_comparison_all.csv')

# LLM encoders
joblib.dump(llm_encoders, output_dir / 'llm_encoders_v4.joblib')

print('Saved all v4 artifacts.')
for f in sorted(output_dir.glob('*v4*')):
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

---
## Wrap up

This was the big jump. Used Claude Haiku to extract 10 structured features from each claim (damage type, severity, incident type, gradual wear flag, etc). F1 macro went from 0.608 to 0.677.

The interesting thing is that LLM features alone (just 10 features) scored 0.642 - that's better than everything from v1 through v3. Turns out 10 interpretable domain-specific features beat 768-dim BERT embeddings.

Top features by SHAP: incident_type (0.63), damage_type (0.28), is_gradual_wear (0.24). These are literally what a human adjuster looks at.

v1: 0.566 --> v2: 0.595 --> v3: 0.608 --> v4: 0.677